# AmSC Infrastructure Resource Orchestration (IRO) Example

AmSC Resource Orchestration Toolkit (AmSCROT) - Orchestrating Infrastructure Service capabilities.

This notebook demonstrates how to use the Toolkit to submit a workflow intent to the AmSC IRO service (SENSE-O) and track its progress and the resulting job outputs.

## Workflow Overview

1. **Initialize** the AmSCROT client
2. **Create** a session
3. **Set up** AmSC IRO ServiceClient (plus additional IRI clients)
4. **Discover** available facilities and intents
5. **Define** Jobs and plan with the desired intent
6. **Plan** and verify the intent
7. **Submit and Monitor**  Job status
8. **View** Outputs
9. **Clean up**

## Prerequisites

### Required Packages
- `amscrot-py` (installed)
- `globus-sdk` (installed)
- `dotenv` (installed)

### Installation

```
pip install amscrot-py globus-sdk dotenv
```

### Credentials

IRI service clients authenticate via API keys stored in `~/.amscrot/credentials.yml` by default. You need entries for each site that will be used for Job submission.

```yaml
# ~/.amscrot/credentials.yml

esnet-iri-east:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://iri-dev.ppg.es.net

esnet-iri-west:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://esnet-west.sdn-sense.net

nersc-iri:
  api_key: <YOUR_BEARER_TOKEN>
  api_endpoint: https://api.iri.nersc.gov

amsc-iro:
  amsc-iro:
  auth_endpoint: https://sense-o-east.es.net:8543/realms/StackV/protocol/openid-connect/token
  api_endpoint: https://sense-o-east.es.net:8443/StackV-web/restapi
  api_key: <YOUR BEARER TOKEN>
  token_issuer: https://auth.globus.org
  client_id: amsc-iro-demo
  secret: <pre-shared IRO secret>
```

In this notebook, the following token auth cells will auto-generate a `~/.amscrot/credentials-new.yml` for you.

---
## 0. Retrieve an authentication token

The process of issuing AmSC tokens is evolving rapidly. For IRI API usage, see the following repository for examples on how to retrieve a token for use with this toolkit.

  * https://github.com/doe-iri/iri-facility-api-examples

Once you have a token, copy it to a `.env` file, set in the `AMSC_TOKEN` environment variable, or paste into the following cell to generate a new credential file for this toolkit example.

In [ ]:
import os
import yaml
from dotenv import load_dotenv

AMSC_TOKEN = None   # <-- manually set token
if not AMSC_TOKEN:
    load_dotenv()   # take environment variables from .env file (if present)
    AMSC_TOKEN = os.getenv("AMSC_TOKEN")

# AmSC IRO service requires a pre-shared IRO ID and SECRET
IRO_ID = os.getenv("IRO_ID")
IRO_SECRET = os.getenv("IRO_SECRET")

# Create credentials data
credentials = {
    'esnet-iri-east': {
        'client_type': 'ESNET_IRI',
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://iri-dev.ppg.es.net'
    },
    'esnet-iri-west': {
        'client_type': 'ESNET_IRI',
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://esnet-west.sdn-sense.net'
    },
    'nersc-iri': {
        'client_type': 'NERSC_IRI',
        'api_key': AMSC_TOKEN,
        'api_endpoint': 'https://api.iri.nersc.gov'
    },
    'amsc-iro': {
        'client_type': 'AMSC_IRO',
        'auth_endpoint': 'https://sense-o-east.es.net:8543/realms/StackV/protocol/openid-connect/token',
        'api_endpoint': 'https://sense-o-east.es.net:8443/StackV-web/restapi',
        'api_key': AMSC_TOKEN, 
        'token_issuer': 'https://auth.globus.org',
        'client_id': IRO_ID,
        'secret': IRO_SECRET
    }
}

# Write to ~/.amscrot/credentials-new.yml
cred_path = os.path.expanduser("~/.amscrot/credentials-new.yml")
os.makedirs(os.path.dirname(cred_path), exist_ok=True)
with open(cred_path, 'w') as f:
    yaml.dump(credentials, f)
print(f"Wrote credentials to {cred_path}")

---
## 1. Initialize Client & Session

In [ ]:
import json
from amscrot.client.client import Client
from amscrot.client.job import Job, JobType, JobServiceType, JobSpec, JobState
from amscrot.model.metadata import Network, Layer2
from amscrot.serviceclient import ServiceClient, PlanError
from amscrot.util.constants import Constants

client = Client(create_service_clients=True,
                credential_file="~/.amscrot/credentials-new.yml")
session = client.create_session("amsc-iro-demo")
print("Client and session initialized.")

## 3.Verify IRI Service Clients

The Client instance just instantiated auto-creates Service Clients for each entrty in the credential file. Let's create convenient handles for each service endpoint for future Job binding.

In [ ]:
iro_client = client.get_service_client("amsc-iro")
iri_east = client.get_service_client("esnet-iri-east")
iri_west = client.get_service_client("esnet-iri-west")

print(f"{iro_client}\n{iri_east}\n{iri_west}")

## 4. Set and find intent and discover resources

Each service client's `discover()` method returns a `DiscoveryResult` container with typed accessors for each resource type. Here we use `.intent` and `.compute` to find available resources at each site.

In [ ]:
# Set our demo intent
intent_name = "AmSC Demo - Networked IRI Jobs - Transfer"

# Discover resources and intents
print("\n--- Discovering Resources & Intents ---")
discovery = iro_client.discover()

facilities = discovery.facility
intents = discovery.intents
print(f"  Facilities: {len(facilities)}")
for f in facilities:
    print(f"    - {f.name} ({f.data.get('id')})")

amsc_intents = [i for i in intents if i.name and "amsc" in i.name.lower()]
print(f"  Intents (AmSC): {len(amsc_intents)}")
for intent in amsc_intents:
    print(f"    - {intent.name} (uuid={intent.uuid}, editable={intent.editable})")

# Resolve a storage resource for filesystem operations
# (prefer a resource with 'home' in its name, fall back to any available)
storage_resources = iri_east.discover().storage or []
storage_resource_id = None
for res in storage_resources:
    if 'home' in (res.data.get('name') or '').lower():
        storage_resource_id = res.data.get('id')
        break
if not storage_resource_id and storage_resources:
    storage_resource_id = storage_resources[0].data.get('id')
print(f"Storage resource ID: {storage_resource_id}")

# Find the Job Placement intent
matching = [i for i in intents if intent_name.lower() in (i.name or "").lower()]
if not matching:
    print(f"\nERROR: No intent matching '{intent_name_}' found.")
    print("Available intents:")
    for i in intents:
        print(f"  - {i.name}")
    sys.exit(1)

selected_intent = matching[0]
print(f"\n--- Selected Intent ---")
print(f"  Name: {selected_intent.name}")
print(f"  UUID: {selected_intent.uuid}")

print (f"\n--- Intent Details --- ")
print (f"{json.dumps(selected_intent.data, indent=2)}")

## 5. Define Job Specs & Jobs

We will run a GPT2 model training compute job. We will also want to launch two transfer containers, one for each facility specified in the selected intent. Note that we don't define the JobSpec details here. Instead, we will let the IRO intent fill in these details for us in the next step. Each Job is annotated with the desired intent UUID to support this mode of Job orchestration.

Note that we can override any editable attributes in the intent using our Toolkit JobSpecs if needed.

In [ ]:
image = "docker.io/dtnaas/tools:oneshot3"

src_path = "/data/home/kissel/results"       # <-- adjust
dst_path = "/data/home/kissel/results"       # <-- adjust
ready_file = "/data/home/kissel/xfer.ready"  # <-- adjust

job1 = Job(
    name="gpt2-compute",
    type=JobType.COMPUTE,
    service_type=JobServiceType.BATCH,
    service_client=iro_client,
    job_spec=JobSpec(),
    intent=selected_intent.uuid
)
session.add_job(job1)

job2 = Job(
    name="transfer-src",
    type=JobType.COMPUTE,
    service_type=JobServiceType.BATCH,
    service_client=iro_client,
    job_spec=JobSpec(executable=f"./run.sh ./wrapper2.sh 2222 {ready_file} {src_path} {dst_path}",
                     attributes={"container": {"image": image}}),
    intent=selected_intent.uuid
)
session.add_job(job2)

job3 = Job(
    name="transfer-dst",
    type=JobType.COMPUTE,
    service_type=JobServiceType.BATCH,
    service_client=iro_client,
    job_spec=JobSpec(executable='./run.sh "./register_ip.sh; sleep 420"',
                     attributes={"container": {"image": image}}),
    intent=selected_intent.uuid
)
session.add_job(job3)

## 6. Plan

The plan phase validates all resources and job specs before anything is created. When we show the session state after plan compeltes, we can see all of the Job specifics populated from the IRO intent.

In [ ]:
try:
    session.plan()
except Exception as e:
    print(f"Failed to plan jobs: {e}")

session.show()

## 7. Apply and monitor

Apply creates resources (if networking is enabled) and submits the compute jobs.
`session.wait()` then polls both jobs until they complete (or raise `WaitTimeoutError`).

In [ ]:
try:
    iri_east.filesystem.rm(storage_resource_id, ready_file)
    iri_east.filesystem.rm(storage_resource_id, src_path)
    iri_west.filesystem.rm(storage_resource_id, dst_path)
except:
    pass

try:
    session.apply()
except Exception as e:
    print(f"Failed to apply session: {e}")
    raise

print("Session applied successfully.")

results = session.wait(
    target_states=[JobState.COMPLETED, JobState.FAILED, JobState.CANCELED],
    timeout=1800,
    interval=2,
    verbose=True,
#    raw=True
)

print(f"\n✅ All jobs completed!")

## 8. View job output

List directories at each facility to see if data is placed where expected.

In [ ]:
if storage_resource_id and iri_east.filesystem:
    try:
        print(f"\n  [EAST] [ls] {src_path} on storage resource {storage_resource_id}:")
        ls_result = iri_east.filesystem.ls(storage_resource_id, src_path, format=True)
        print(f"  {ls_result}")
    except Exception as ls_err:
        print(f"  [ls] failed: {ls_err}")

if storage_resource_id and iri_west.filesystem:
    try:
        print(f"\n  [WEST] [ls] {dst_path} on storage resource {storage_resource_id}:")
        ls_result = iri_west.filesystem.ls(storage_resource_id, dst_path, format=True)
        print(f"  {ls_result}")
    except Exception as ls_err:
        print(f"  [ls] failed: {ls_err}")

## 9. Clean Up

Destroy the session to tear down any provisioned resources and cancel remaining jobs.

In [ ]:
session.destroy()